In [1]:
from ansys.mapdl.core import launch_mapdl
from pathlib import Path
import pandas as pd
import numpy as np

# --------------------
# Paths
# --------------------
run_dir = Path("C:/temp")
csv_file = Path(
    r"C:\github\ICME-OoCTO\results\clustering\MTS_example1_25x25.xls_interp_97x97_GMM_2026-01-07_14h05\data_with_clusters_and_phases.csv"
)
run_dir.mkdir(exist_ok=True)

# --------------------
# Read CSV
# --------------------
df = pd.read_csv(csv_file)

# Keep only what we need
df = df[["X Position_µm", "Y Position_µm", "Phase"]]
df.columns = ["x", "y", "phase"]

# --------------------
# Build grid
# --------------------
xs = np.sort(df.x.unique())
ys = np.sort(df.y.unique())

NX = len(xs)
NY = len(ys)
NZ = 1

DX = np.mean(np.diff(xs))
DY = np.mean(np.diff(ys))

Lx = xs.max() - xs.min()
DZ = 0.1 * Lx

print(f"Grid: {NX} x {NY} x {NZ}")
print(f"DX={DX:.4f} µm, DY={DY:.4f} µm, DZ={DZ:.4f} µm")

# Create index lookup
x_id = {x: i for i, x in enumerate(xs)}
y_id = {y: j for j, y in enumerate(ys)}

# --------------------
# Launch MAPDL
# --------------------
mapdl = launch_mapdl(
    run_location=str(run_dir),
    jobname="rve_test",
    override=True
)

mapdl.clear()
mapdl.prep7()

mapdl.et(1, 185)  # SOLID185

# --------------------
# Nodes
# --------------------
node_id = 0
node_map = {}

for k, z in enumerate([0.0, DZ]):
    for j, y in enumerate(ys):
        for i, x in enumerate(xs):
            node_id += 1
            mapdl.n(node_id, x, y, z)
            node_map[(i, j, k)] = node_id

def nid(i, j, k):
    return node_map[(i, j, k)]

# --------------------
# Elements (1 layer in Z)
# --------------------
elem_id = 0
elem_map = {}

for j in range(NY - 1):
    for i in range(NX - 1):
        elem_id += 1
        mapdl.e(
            nid(i,   j,   0),
            nid(i+1, j,   0),
            nid(i+1, j+1, 0),
            nid(i,   j+1, 0),
            nid(i,   j,   1),
            nid(i+1, j,   1),
            nid(i+1, j+1, 1),
            nid(i,   j+1, 1),
        )
        elem_map[(i, j)] = elem_id

# --------------------
# Materials from phases
# --------------------
phases = sorted(df.phase.unique())

for mid, phase in enumerate(phases, start=1):
    mapdl.mp("EX", mid, 200e3)     # placeholder
    mapdl.mp("NUXY", mid, 0.3)
    mapdl.mp("DENS", mid, 7.8e-9)

phase_to_mat = {p: i+1 for i, p in enumerate(phases)}

# --------------------
# Assign materials per element
# --------------------
for _, row in df.iterrows():
    i = x_id[row.x]
    j = y_id[row.y]

    # Skip last row/column (no element there)
    if i == NX-1 or j == NY-1:
        continue

    e = elem_map[(i, j)]
    mapdl.emodif(e, "MAT", phase_to_mat[row.phase])

# --------------------
# Save
# --------------------
mapdl.save()
mapdl.cdwrite("DB", "rve_test", "cdb")

mapdl.finish()
mapdl.exit()


Grid: 97 x 97 x 1
DX=0.5155 µm, DY=0.5155 µm, DZ=4.9485 µm


In [2]:
from ansys.mapdl.core import launch_mapdl

mapdl = launch_mapdl()

mapdl.run("/CWD,'C:/temp'")

mapdl.clear()
mapdl.prep7()
mapdl.cdread("DB", "rve_test", "cdb")

mapdl.allsel()

# APDL plot (works)
mapdl.eplot(color_by="mat", show_edges=True)

# ---- PyVista: robust material extraction ----
grid = mapdl.mesh.grid
elem_ids = grid.cell_data["ansys_elem_num"]

mat = [mapdl.get("M", "ELEM", int(eid), "ATTR", "MAT") for eid in elem_ids]

grid.cell_data["mat"] = mat
grid.plot(scalars="mat", show_edges=True)


[82, 87, 110]


Widget(value='<iframe src="http://localhost:53191/index.html?ui=P_0x2039786e4d0_0&reconnect=auto" class="pyvis…

Widget(value='<iframe src="http://localhost:53191/index.html?ui=P_0x2039f0ff290_1&reconnect=auto" class="pyvis…